# 01 - Pricing EDA (interactive companion)

This notebook is a **companion** to `scripts/run_eda.py`, which is the source of
truth for every number in `reports/02_PRICING_EDA.md`. Use the notebook to explore,
the script to produce reproducible artifacts.

Run `make data && make eda` first.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pricing_engine.config import load_config
cfg = load_config(ROOT / "configs" / "config.yaml")
pd.set_option("display.width", 140)


In [ ]:
df = pd.read_parquet(cfg.path("processed_table"))
print(f"{len(df):,} canonical rows | {df['upc'].nunique()} UPCs | "
      f"{df['store'].nunique()} stores | {df['week'].nunique()} weeks")
df.head()

## 1. Historical scale (observed, not modelled)

In [ ]:
print("observed revenue         :", f"${df['revenue'].sum():,.0f}")
print("observed gross profit    :", f"${df['gross_profit'].sum():,.0f}")
print("overall gross margin     :", f"{100*df['gross_profit'].sum()/df['revenue'].sum():.2f}%")
print("mean effective unit price:", f"${df['effective_unit_price'].mean():.3f}")

## 2. Where the gross profit sits

In [ ]:
by_upc = (df.groupby("upc")
            .agg(descrip=("descrip", "first"), units=("move", "sum"),
                 revenue=("revenue", "sum"), gross_profit=("gross_profit", "sum"))
            .sort_values("gross_profit", ascending=False))
by_upc.head(15)

## 3. Within-series price variation - the eligibility question

A series whose price never moved cannot support a price recommendation, no matter
how good the demand model is.

In [ ]:
from pricing_engine.economics.metrics import price_variation_summary, eligible_series

var = price_variation_summary(df, ["upc", "store"])
var["eligible"] = eligible_series(var, cfg.require("eligibility"))
print(f"{int(var['eligible'].sum()):,} of {len(var):,} series eligible "
      f"({100*var['eligible'].mean():.1f}%)")
var[["n_obs", "n_distinct_prices", "price_cv", "n_price_changes"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(var["price_cv"].clip(upper=0.4), bins=60)
ax.axvline(cfg.get("eligibility.min_price_cv"), color="crimson", ls="--")
ax.set_title("Within UPC x store price coefficient of variation")
plt.show()

## 4. Price vs demand - association only

The scatter below is **not** a demand curve: price and demand are jointly determined
by promotions, seasonality and retailer decisions. See `docs/CAUSAL_LIMITATIONS.md`.

In [ ]:
s = df.sample(60_000, random_state=cfg.seed)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(s["effective_unit_price"], np.log1p(s["move"]), s=1, alpha=0.05)
ax.set_xlim(0, 8); ax.set_xlabel("effective unit price ($)"); ax.set_ylabel("log(1 + units)")
plt.show()

## 5. Explore one series

In [ ]:
upc, store = int(by_upc.index[0]), int(df["store"].mode()[0])
one = df[(df["upc"] == upc) & (df["store"] == store)].sort_values("week")
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(one["week_start_date"], one["effective_unit_price"], label="price")
ax2 = ax.twinx()
ax2.bar(one["week_start_date"], one["move"], alpha=0.3, width=5, color="grey")
ax.set_title(f"{one['descrip'].iloc[0]} - store {store}")
ax.legend()
plt.show()